## Prerequisites

As in Tutorial 7.1, we'll use OpenMM to run molecular dynamics and MDTraj to compute the progress coordinate. To install OpenMM, run the following cell:

In [ ]:
!pip install openmm

To configure the simulation, we'll need two input files:
1. `topology.pdb` &mdash; PDB file specifying the molecular topology
2. `initial_state.xml` &mdash; OpenMM XML file containing the initial (unbound) state

To clean up output files generated by previous runs, run the following cell:

In [2]:
import os
import shutil

if os.path.exists('west.h5'):
    os.remove('west.h5')
if os.path.exists('traj_segs'):
    shutil.rmtree('traj_segs')

The API makes use of Python's `logging` module for event logging. For that output to appear in the notebook, we need to direct it to stdout:

In [3]:
import logging
import sys

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

## Propagator

The `OpenMMPropagator` class provides a ready-to-use interface to the OpenMM application layer. To create an instance, we need the same inputs needed to create an OpenMM `Simulation` instance. At a minimum, this includes a topology, a parametrized system, and an integrator:

In [4]:
import openmm.app
from openmm import unit

forcefield = openmm.app.ForceField('amber14-all.xml', 'amber14/tip3p.xml')

topology = openmm.app.PDBFile('topology.pdb').getTopology()
system = forcefield.createSystem(
    topology,
    nonbondedMethod=openmm.app.PME,
    nonbondedCutoff=1 * unit.nanometer,
    constraints=openmm.app.HBonds,
)
system.addForce(openmm.MonteCarloBarostat(1 * unit.bar, 300 * unit.kelvin))
integrator = openmm.LangevinIntegrator(
    300 * unit.kelvin, 1 / unit.picosecond, 2 * unit.femtosecond,
)

In addition, we need to specify the length of each trajectory segment. This is done using the `steps` parameter:

In [5]:
propagator = westpa.OpenMMPropagator(
    topology=topology,
    system=system,
    integrator=integrator,
    steps=1000,
)

INFO:westpa.core.propagators.base:root_seed=225916294099962807911271622052060578881


By default, the propagator only outputs the final state of each segment. To report additional output, use the `add_reporter()` method:

In [6]:
propagator.add_reporter(
    openmm.app.XTCReporter,
    filename='traj.xtc',
    report_interval=500,
)
propagator.add_reporter(
    openmm.app.StateDataReporter,
    filename='log.csv',
    report_interval=100,
    options=dict(step=True, potentialEnergy=True, kineticEnergy=True, temperature=True),
)

## Progress coordinate

To specify the progress coordinate, we define a function that takes a `Segment` object, sets its `pcoord` attribute, and returns the modified object:

In [7]:
import mdtraj

top = mdtraj.Topology.from_openmm(topology)

def pcoord_calculator(segment):
    traj = mdtraj.load_xml(segment.final_state.file, top=top)
    distances = mdtraj.compute_distances(traj, atom_pairs=[[0, 1]])
    segment.pcoord = distances * 10  # nanometer -> angstrom
    return segment

Note that we only compute the Na+/Cl- distance for the final state (i.e., `pcoord_len == 1`), since only this point is needed for resampling.

## Initializing and running the simulation

Finally, we can create, initialize, and run the simulation:

In [8]:
import os.path
import numpy as np

initial_state = westpa.State(file=os.path.abspath('initial_state.xml'))

simulation = westpa.Simulation(
    propagator=propagator,
    pcoord_calculator=pcoord_calculator,
    bin_mapper=westpa.RectilinearBinMapper(
        boundaries=[
            [0, 2.6, 2.8, 3, 3.2, 3.4, 3.6, 3.8, 4, 4.5, 5, 5.5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, np.inf]
        ],
    ),
    bin_target_counts=5,
    source=westpa.Source([initial_state]),
    sinks=westpa.Sink(indicator=lambda seg: seg.pcoord[-1, 0] < 2.6, label='bound')
)

INFO:westpa.core.resamplers.base:Using NumPy default_rng(seed=98798557279766400133699440587284379047)


In [9]:
simulation.initialize([initial_state] * 5)

INFO:westpa.core.simulation:Created HDF5 file 'west.h5'
INFO:westpa.core.simulation:Simulation prepared.
INFO:westpa.core.simulation:number of segments:             5
INFO:westpa.core.simulation:minimum non-zero probability:   0.2
INFO:westpa.core.simulation:maximum non-zero probability:   0.2
INFO:westpa.core.simulation:probability dynamic range (kT): 0
INFO:westpa.core.simulation:norm = 1, error in norm = 0 (0 * eps)


In [ ]:
simulation.run(3)